# Can a model trained on TCGA reach 0.815 AUC in CPTAC-3?

This executed experiment predicts known two-year mortality in lung adenocarcinoma. All fitting and parameter selection use the paper's 267 TCGA patients. The main transfer test uses the 85 CPTAC-3 patients analyzed in the paper, with additional checks in two GEO studies. These 85 patients have been examined in an earlier pilot. The current model, comparison arms and evaluation rules were specified before scoring the new fits. This is exploratory replication and external validation of a search direction. A formal prospective test remains open.

**Google Colab**: select **Runtime → Run all** on a CPU. The inputs are bundled, and model training is repeated when this notebook runs. It writes patient-level scores and metrics as CSV files in the working directory.

## What changed

The primary model ranks patients by major pathological stage. Among patients at the same stage it uses the fitted two-year mortality score from finer stage, age, sex and RNA expression. The RNA feature counts the proportion of 25 comparisons in which a proliferation gene is expressed more strongly than a lung differentiation gene in the same sample. Ties count as one half. The ordering is invariant to a common monotone transformation within a sample, although gene-specific assay effects can still alter it. Gene selection is a research hypothesis.

The clinical control uses the same rule without RNA. Additional comparisons include age/sex/substage plus TNM, the ten-gene score alone, and the original paper’s 32 molecular features ranked within each patient and assay. We use training medians to fill missing values and five tissue-source-grouped folds inside TCGA to select the ridge penalty. The final stage-based score is ordinal. This experiment evaluates ranking and average precision. It does not produce a calibrated mortality probability.

The first pilot showed stage transported better than the original molecular random forest. That informed this follow-up. All comparison arms are disclosed, including assays unavailable for the GEO cohorts.

In [1]:
#@title Load the measured data and fixed experiment specification
import base64, hashlib, io, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
blob = base64.b64decode("""
...
""")
assert hashlib.sha256(blob).hexdigest() == "da1eddca50cc2e493a76e2f62bd71f5507e5d9b3ea2a3ae828892eeeb7f49523"
archive = zipfile.ZipFile(io.BytesIO(blob))
with archive.open("measurements.csv") as f:
    data = pd.read_csv(f, index_col="patient_id",float_precision="round_trip")
panel = json.loads(archive.read("shared_panel.json"))
assert data.index.is_unique and len(panel) == 32
print(data.groupby("cohort").death.agg(["size","sum"]).rename(columns={"size":"evaluable patients","sum":"deaths"}).to_string())

          evaluable patients  deaths
cohort                              
CPTAC-3                   96      12
GSE31210                 201      11
GSE72094                 320      78
TCGA                     267      89


## Train using TCGA only

The original paper reported 0.815 after fitting and testing within CPTAC-3. It reported poor direct transfer qualitatively. The 0.815 line is therefore a numerical target here. It does not represent the paper’s direct-transfer AUC. The fixed models below are applied to each external cohort without any target-driven normalization, tuning, calibration, or retraining. The GEO matrices use the preprocessing supplied by their depositors.

In [2]:
"""Frozen TCGA models evaluated on patient-level external cohorts."""
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
SEED = 20261125
GROWTH = ['MKI67','TOP2A','BIRC5','CCNB1','UBE2C']
LUNG = ['SFTPB','SFTPC','SCGB1A1','NAPSA','SLC34A2']
STAGE = 'Major stage'
CLIN = 'Stage + clinical'
PRIMARY = 'Stage + clinical + ten-gene RNA'
RNA = 'Ten-gene RNA only'
PANEL = 'Original 32, ranked, molecular only'
WITH_PANEL = 'Stage + clinical + original 32, ranked'
TNM = 'Stage + clinical + TNM'
TNM_RNA = 'Stage + clinical + TNM + ten-gene RNA'

def features(data, shared_panel):
    x = data.copy()
    dif = x[GROWTH].to_numpy()[:,:,None] - x[LUNG].to_numpy()[:,None,:]
    valid = np.isfinite(dif)
    x['rna_pairs'] = np.divide(((dif>0) + 0.5*(dif==0)).sum(axis=(1,2)),valid.sum(axis=(1,2)),
                                out=np.full(len(x),np.nan),where=valid.sum(axis=(1,2))>0)
    for is_cpg in [True,False]:
        cols=[c for c in shared_panel if c.startswith('cg')==is_cpg]
        ranks = x[cols].rank(axis=1,pct=True,method='average')
        x[['rank_'+c for c in cols]] = ranks
    return x

def definitions(shared_panel):
    ranks=['rank_'+c for c in shared_panel]
    clinical=['major','sub','age','male']
    return {CLIN:clinical,PRIMARY:clinical+['rna_pairs'],RNA:['rna_pairs'],PANEL:ranks,
            WITH_PANEL:clinical+ranks,TNM:clinical+['T','N','M'],TNM_RNA:clinical+['T','N','M','rna_pairs']}

def develop(train,shared_panel):
    assert train.cohort.eq('TCGA').all() and train.index.is_unique
    groups=train.index.str.split('-').str[1]
    splits=list(GroupKFold(5).split(train,train.death,groups=groups))
    models={}
    for name,cols in definitions(shared_panel).items():
        if not train[cols].notna().any().all():
            raise ValueError('A declared training feature has no measurements: '+name)
        pipeline=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),
                               LogisticRegression(solver='lbfgs',max_iter=3000))
        search=GridSearchCV(pipeline,{'logisticregression__C':[0.01,0.1,1.0,10.0]},
                            cv=splits,scoring='neg_log_loss',error_score='raise',n_jobs=1)
        search.fit(train[cols],train.death)
        models[name]=(cols,search.best_estimator_,search.best_params_['logisticregression__C'])
    return models,float(train.major.median())

def score(models, stage_fill, target, molecular_panel_available):
    scores=pd.DataFrame(index=target.index)
    scores[STAGE]=target.major.fillna(stage_fill)
    for name,(columns,model,_) in models.items():
        if not molecular_panel_available and name in (PANEL,WITH_PANEL,TNM,TNM_RNA):continue
        within=model.predict_proba(target[columns])[:,1]
        scores[name]=within if name in (RNA,PANEL) else scores[STAGE]+within
    return scores

def evaluate(scores,y,cohort_name,reps=10000):
    assert scores.index.equals(y.index) and scores.notna().all().all()
    positive=scores.loc[y.eq(1)].to_numpy()
    negative=scores.loc[y.eq(0)].to_numpy()
    delta=positive[:,None,:]-negative[None,:,:]
    wins=(delta>0).astype(float)+.5*(delta==0)
    point=wins.mean(axis=(0,1))
    assert np.allclose(point,[roc_auc_score(y,scores[name]) for name in scores])
    n1,n0=len(positive),len(negative)
    rng=np.random.default_rng(SEED)
    # Use batches to bound memory. Resample whole patients and share draws across arms.
    boots=[]
    for start in range(0,reps,250):
        n=min(250,reps-start)
        a=rng.multinomial(n1,np.repeat(1/n1,n1),size=n)
        b=rng.multinomial(n0,np.repeat(1/n0,n0),size=n)
        boots.append(np.einsum('bi,ijm,bj->bm',a,wins,b,optimize=True)/(n1*n0))
    boots=np.concatenate(boots)
    results=[]
    for i,name in enumerate(scores):
        row={'cohort':cohort_name,'model':name,'patients':len(y),'deaths':int(y.sum()),
             'auc':point[i],'auc_low':np.quantile(boots[:,i],.025),'auc_high':np.quantile(boots[:,i],.975),
             'average_precision':average_precision_score(y,scores[name])}
        for label,base in [('stage',STAGE),('clinical',CLIN)]:
            j=scores.columns.get_loc(base)
            row['delta_vs_'+label]=point[i]-point[j]
            row['delta_vs_'+label+'_low'],row['delta_vs_'+label+'_high']=np.quantile(boots[:,i]-boots[:,j],[.025,.975])
        results.append(row)
    return pd.DataFrame(results)

In [3]:
x = features(data, panel)
train = x.loc[x.cohort.eq('TCGA')]
assert len(train)==267 and train.death.sum()==89
models, stage_median = develop(train,panel)
print('Regularization chosen by TCGA site-grouped validation')
print({name: C for name,(_,_,C) in models.items()})

Regularization chosen by TCGA site-grouped validation
{'Stage + clinical': 0.1, 'Stage + clinical + ten-gene RNA': 0.1, 'Ten-gene RNA only': 0.1, 'Original 32, ranked, molecular only': 0.01, 'Stage + clinical + original 32, ranked': 0.01, 'Stage + clinical + TNM': 0.1, 'Stage + clinical + TNM + ten-gene RNA': 0.1}


## External results

The CPTAC primary set matches the base paper. The 96-case sensitivity includes 11 additional eligible patients. These two sets overlap. External bootstrap intervals condition on the fitted TCGA models and resample patients separately within the two outcome classes. Paired intervals use the same patients and draws for both models. AUC in early-stage-only cohorts may be especially uncertain when deaths are rare.

In [4]:
summaries, rankings = [], []
for cohort, label, use_paper_ids in [('CPTAC-3','CPTAC-3 primary',True),('GSE72094','GSE72094',True),
                                      ('GSE31210','GSE31210',True),('CPTAC-3','CPTAC-3 all 96',False)]:
    target = x.loc[x.cohort.eq(cohort)]
    if use_paper_ids:
        target = target.loc[target.primary_set]
    scores = score(models,stage_median,target,cohort=='CPTAC-3')
    assert scores.index.is_unique and set(train.index).isdisjoint(target.index)
    rankings.append(scores.assign(test=label,death=target.death))
    summaries.append(evaluate(scores,target.death,label))
result = pd.concat(summaries,ignore_index=True)
predictions = pd.concat(rankings)
result.to_csv('LUAD_transfer_metrics.csv',index=False)
predictions.to_csv('LUAD_transfer_patient_scores.csv')
print(result.loc[result.model.eq(PRIMARY),['cohort','patients','deaths','auc','auc_low','auc_high','delta_vs_clinical','delta_vs_clinical_low','delta_vs_clinical_high']].round(4).to_string(index=False))

         cohort  patients  deaths    auc  auc_low  auc_high  delta_vs_clinical  delta_vs_clinical_low  delta_vs_clinical_high
CPTAC-3 primary        85      10 0.7547   0.5733    0.9040             0.0120                -0.0053                  0.0347
       GSE72094       320      78 0.6911   0.6243    0.7565             0.0229                 0.0076                  0.0402
       GSE31210       201      11 0.9481   0.9038    0.9823            -0.0012                -0.0196                  0.0179
 CPTAC-3 all 96        96      12 0.7679   0.6151    0.8958             0.0089                -0.0069                  0.0288


In [5]:
print(result.pivot(index='model',columns='cohort',values='auc').round(4).to_string())
print('Fraction of each cohort with recorded baseline inputs')
print(x.groupby('cohort')[['major','sub','age','male','T','N','M']].agg(lambda c: c.notna().mean()).round(3).to_string())

cohort                                  CPTAC-3 all 96  CPTAC-3 primary  GSE31210  GSE72094
model                                                                                      
Major stage                                     0.7847           0.7880    0.8730    0.6536
Original 32, ranked, molecular only             0.5228           0.5747       NaN       NaN
Stage + clinical                                0.7589           0.7427    0.9493    0.6682
Stage + clinical + TNM                          0.7470           0.7307       NaN       NaN
Stage + clinical + TNM + ten-gene RNA           0.7480           0.7307       NaN       NaN
Stage + clinical + original 32, ranked          0.7688           0.7773       NaN       NaN
Stage + clinical + ten-gene RNA                 0.7679           0.7547    0.9481    0.6911
Ten-gene RNA only                               0.6255           0.6353    0.6624    0.6692
Fraction of each cohort with recorded baseline inputs
          major    sub    

## Interpreting this run

| Test set | Patients | Deaths | Primary AUC | 95% interval |
|---|---:|---:|---:|---|
| CPTAC-3 primary | 85 | 10 | 0.7547 | 0.5733–0.9040 |
| GSE72094 | 320 | 78 | 0.6911 | 0.6243–0.7565 |
| GSE31210 | 201 | 11 | 0.9481 | 0.9038–0.9823 |
| CPTAC-3 all 96 | 96 | 12 | 0.7679 | 0.6151–0.8958 |

Judge the molecular contribution by the paired difference versus the otherwise identical clinical model. The stage-based and extended models use information outside the paper's original 32 markers. The GEO cohorts contain RNA and lack the corresponding methylation panel. We therefore evaluated the clinically anchored ten-gene model and its applicable controls there. Cohort names and sample identifiers differ across repositories, so full identities cannot be cross-linked. Absolute-risk calibration, censoring before two years, and harmonization of cause-of-death definitions are priorities for a publication study.

## Data and method provenance

- The original study supplies TCGA/CPTAC-3 patient IDs, endpoint labels and feature names. CPTAC excludes certain non-cancer deaths, while TCGA lacks equivalent cause-of-death information. [Base paper and supplements](https://doi.org/10.3390/ijms25073661)
- GDC supplies stage, T/N/M and age for the primary diagnosis, and recorded sex at birth. Tumor RNA uses UCSC Xena GDC STAR TPM. When several primary tumors match one patient, gene values are combined by their median. Original methylation and RNA inputs are taken from the preceding, audited reconstruction. SHAP or survival-derived variables are never included as predictors. [GDC](https://portal.gdc.cancer.gov/) · [UCSC Xena](https://github.com/ucscXena/xena-GDC-ETL)
- GSE72094 uses the deposited GPL15048 expression matrix, the submitted stage and overall survival fields. GSE31210 uses GPL570, primary tumors, and the depositor’s prognosis-analysis eligibility flag. For both studies the analysis includes a death before day 730 or known survival to day 730. Earlier censoring is excluded. Time origin can differ between diagnosis and resection. Gene measures are medians of unambiguously named probes, using the platform tables. Sample and probe mappings are included in the bundled data. [GSE72094](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE72094) · [GSE31210](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE31210)
- Existing lung adenocarcinoma work has explored single-sample expression ordering and molecular subtype prediction. This experiment does not establish a first-in-field method. [Prior study](https://pubmed.ncbi.nlm.nih.gov/32745259/)

The bundled protocol records the models and numeric targets. This notebook was executed with Python 3.12, pandas 2.2.3 and scikit-learn 1.8.0.